In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "gpt2"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, output_hidden_states=True)

text = "The capital of Italy is"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

hidden_states = outputs.hidden_states
lm_head = model.lm_head.weight

print("Layer predictions:")

for layer_idx, h in enumerate(hidden_states):

    last_token = h[0, -1, :]
    logits = lm_head @ last_token
    probs = torch.softmax(logits, dim=-1)

    top_prob, top_idx = torch.topk(probs, 1)
    token = tokenizer.decode(top_idx)

    print(f"Layer {layer_idx}: {token} ({top_prob.item():.3f})")

In [ ]:
import os
os.makedirs("/kaggle/working/plots", exist_ok=True)

In [ ]:
model_name = "gpt2"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, output_hidden_states=True)

text = "The capital of Italy is"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

hidden_states = outputs.hidden_states
lm_head = model.lm_head.weight

print("Layer predictions:")

for layer_idx, h in enumerate(hidden_states):

    last_token = h[0, -1, :]
    logits = lm_head @ last_token
    probs = torch.softmax(logits, dim=-1)

    top_probs, top_indices = torch.topk(probs, 5)

    tokens = [tokenizer.decode([i]) for i in top_indices]

    print(f"Layer {layer_idx}: {list(zip(tokens, top_probs.tolist()))}")

In [ ]:
rome_id = tokenizer.encode(" Rome")[0]

rome_probs = []

for layer_idx, h in enumerate(hidden_states):

    last_token = h[0, -1, :]
    logits = lm_head @ last_token
    probs = torch.softmax(logits, dim=-1)

    rome_prob = probs[rome_id].item()

    rome_probs.append(rome_prob)

    print(f"Layer {layer_idx}: P(Rome) = {rome_prob}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(len(rome_probs)), rome_probs, marker="o")
plt.xlabel("Layer")
plt.ylabel("Probability of 'Rome'")
plt.title("Emergence of 'Rome' Across Transformer Layers")
plt.show()

In [ ]:
targets = [" Rome", " Milan", " Paris"]

target_ids = [tokenizer.encode(t)[0] for t in targets]

for layer_idx, h in enumerate(hidden_states):

    last_token = h[0, -1, :]
    logits = lm_head @ last_token
    probs = torch.softmax(logits, dim=-1)

    print(f"\nLayer {layer_idx}")

    for t, tid in zip(targets, target_ids):
        print(t, probs[tid].item())

In [ ]:
prompts = {
    "The capital of Italy is": " Rome",
    "The capital of France is": " Paris",
    "The capital of Germany is": " Berlin",
    "The capital of Spain is": " Madrid",

    "The largest planet in the solar system is": " Jupiter",
    "The fastest land animal is": " cheetah",

    "The author of Hamlet is": " Shakespeare",
    "The theory of relativity was proposed by": " Einstein",

    "The tallest mountain in the world is": " Everest",
    "The chemical symbol for water is": " H"
}

In [ ]:
for prompt, answer in prompts.items():

    print("\n========================")
    print("Prompt:", prompt)

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    hidden_states = outputs.hidden_states
    answer_id = tokenizer.encode(answer)[0]

    probs_list = []

    for layer_idx, h in enumerate(hidden_states):

        last_token = h[0, -1, :]
        logits = lm_head @ last_token
        probs = torch.softmax(logits, dim=-1)

        prob = probs[answer_id].item()

        probs_list.append(prob)

        print(f"Layer {layer_idx}: P({answer}) = {prob}")

    layers = list(range(len(probs_list)))

    plt.figure(figsize=(6,4))
    plt.plot(layers, probs_list, marker="o")

    plt.xlabel("Layer")
    plt.ylabel(f"P({answer})")
    plt.title(prompt)

    filename = f"/kaggle/working/plots/{answer.strip()}_layer_plot.png"

    plt.savefig(filename)
    plt.close()

    print("Saved plot:", filename)